In [1]:
import sys
from pathlib import Path

PROJECT_ROOT = Path.cwd().resolve().parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.append(str(PROJECT_ROOT))

print("Project root added to path:", PROJECT_ROOT)

Project root added to path: C:\dev\util


In [3]:
from src.data_fetcher import build_live_carbon_forecast_table

df = build_live_carbon_forecast_table(region="CAISO")
print(df.head())
print(df.columns)

FORECAST STATUS: 200
FORECAST CONTENT TYPE: application/json
FORECAST URL: https://api.watttime.org/v3/forecast?region=CAISO_NORTH&signal_type=co2_moer
FORECAST RESPONSE PREVIEW: {"data":[{"point_time":"2026-03-14T03:45:00+00:00","value":997.2},{"point_time":"2026-03-14T03:50:00+00:00","value":997.2},{"point_time":"2026-03-14T03:55:00+00:00","value":997.0},{"point_time":"2026-03-14T04:00:00+00:00","value":993.9},{"point_time":"2026-03-14T04:05:00+00:00","value":992.1},{"point_time":"2026-03-14T04:10:00+00:00","value":992.4},{"point_time":"2026-03-14T04:15:00+00:00","value":992.9},{"point_time":"2026-03-14T04:20:00+00:00","value":988.9},{"point_time":"2026-03-14T04:25:00+
                  timestamp  carbon_g_per_kwh  price_per_kwh
0 2026-03-14 03:45:00+00:00             997.2           0.15
1 2026-03-14 03:50:00+00:00             997.2           0.15
2 2026-03-14 03:55:00+00:00             997.0           0.15
3 2026-03-14 04:00:00+00:00             993.9           0.15
4 2026-03-14 04

In [2]:
from src.inputs import WorkloadInput
from src.pipeline import run_util_pipeline

In [9]:
from src.pipeline import run_util_pipeline
import inspect

print(run_util_pipeline)
print(inspect.signature(run_util_pipeline))
print(run_util_pipeline.__code__.co_filename)

<function run_util_pipeline at 0x00000134BAC3A8E0>
(workload_input: 'Any', mapping_path: 'str | Path', carbon_path: 'str | Path | None' = None, price_path: 'str | Path | None' = None, forecast_mode: 'str' = 'demo') -> 'dict[str, Any]'
C:\dev\util\src\pipeline.py


In [10]:
result = run_util_pipeline(
    workload_input=workload,
    mapping_path="data/raw/zip_to_region_sample.csv",
    carbon_path="data/raw/sample_carbon_forecast.csv",
    price_path="data/raw/sample_price_forecast.csv",
    forecast_mode="demo",
)

FileNotFoundError: [Errno 2] No such file or directory: 'data/raw/zip_to_region_sample.csv'

In [3]:
result = run_util_pipeline(
    workload_input=workload,
    mapping_path=PROJECT_ROOT / "data" / "raw" / "zip_to_region_sample.csv",
    carbon_path=PROJECT_ROOT / "data" / "raw" / "sample_carbon_forecast.csv",
    price_path=PROJECT_ROOT / "data" / "raw" / "sample_price_forecast.csv",
)

NameError: name 'workload' is not defined

In [ ]:
import sys
from pathlib import Path

NOTEBOOK_DIR = Path.cwd().resolve()

if NOTEBOOK_DIR.name == "notebooks":
    PROJECT_ROOT = NOTEBOOK_DIR.parent
else:
    PROJECT_ROOT = NOTEBOOK_DIR

if str(PROJECT_ROOT) not in sys.path:
    sys.path.append(str(PROJECT_ROOT))

print("Notebook directory:", NOTEBOOK_DIR)
print("Project root:", PROJECT_ROOT)

Notebook directory: C:\dev\util\notebooks
Project root: C:\dev\util


In [ ]:
# -------------------------------------------------
# IMPORT UTIL MODULES
# -------------------------------------------------

from src.data_fetcher import build_forecast_table
from src.baseline import build_baseline_schedule
from src.optimizer import optimize_schedule
from src.metrics import compare_schedules

In [ ]:
# -------------------------------------------------
# LOAD PLACEHOLDER FORECAST DATA
# -------------------------------------------------

forecast_df = build_forecast_table(carbon_path, price_path)

print("Rows in forecast:", len(forecast_df))
forecast_df.head(10)

Rows in forecast: 24


,timestamp,carbon_g_per_kwh,price_per_kwh
0,2026-03-13 00:00:00,340,0.09
1,2026-03-13 01:00:00,330,0.08
2,2026-03-13 02:00:00,320,0.08
3,2026-03-13 03:00:00,310,0.07
4,2026-03-13 04:00:00,300,0.07
5,2026-03-13 05:00:00,290,0.08
6,2026-03-13 06:00:00,280,0.10
7,2026-03-13 07:00:00,270,0.13
8,2026-03-13 08:00:00,220,0.16
9,2026-03-13 09:00:00,210,0.18


In [ ]:
baseline_df = build_baseline_schedule(
    forecast_df=forecast_df,
    compute_hours_required=8
)

baseline_df

,timestamp,carbon_g_per_kwh,price_per_kwh,baseline_run_flag
0,2026-03-13 00:00:00,340,0.09,1
1,2026-03-13 01:00:00,330,0.08,1
2,2026-03-13 02:00:00,320,0.08,1
3,2026-03-13 03:00:00,310,0.07,1
4,2026-03-13 04:00:00,300,0.07,1
5,2026-03-13 05:00:00,290,0.08,1
6,2026-03-13 06:00:00,280,0.10,1
7,2026-03-13 07:00:00,270,0.13,1
8,2026-03-13 08:00:00,220,0.16,0
9,2026-03-13 09:00:00,210,0.18,0


In [ ]:
baseline_df = build_baseline_schedule(
    forecast_df=forecast_df,
    compute_hours_required=8
)

optimized_df = optimize_schedule(
    forecast_df=forecast_df,
    compute_hours_required=8,
    objective="carbon"
)

optimized_df

,timestamp,carbon_g_per_kwh,price_per_kwh,score,run_flag
0,2026-03-13 00:00:00,340,0.09,340,0
1,2026-03-13 01:00:00,330,0.08,330,0
2,2026-03-13 02:00:00,320,0.08,320,0
3,2026-03-13 03:00:00,310,0.07,310,0
4,2026-03-13 04:00:00,300,0.07,300,0
5,2026-03-13 05:00:00,290,0.08,290,0
6,2026-03-13 06:00:00,280,0.10,280,0
7,2026-03-13 07:00:00,270,0.13,270,0
8,2026-03-13 08:00:00,220,0.16,220,0
9,2026-03-13 09:00:00,210,0.18,210,0


In [ ]:
comparison = compare_schedules(
    baseline_df=baseline_df,
    optimized_df=optimized_df,
    machine_watts=300
)

import pandas as pd
comparison_df = pd.DataFrame([comparison])

comparison_df

,baseline_cost,optimized_cost,cost_savings,cost_reduction_pct,baseline_carbon_kg,optimized_carbon_kg,carbon_savings_kg,carbon_reduction_pct
0,0.21,0.678,-0.468,-222.857143,0.732,0.378,0.354,48.360656


In [ ]:
optimized_df[optimized_df["run_flag"] == 1][["timestamp","carbon_g_per_kwh"]]

,timestamp,carbon_g_per_kwh
10,2026-03-13 10:00:00,200
11,2026-03-13 11:00:00,180
12,2026-03-13 12:00:00,160
13,2026-03-13 13:00:00,150
14,2026-03-13 14:00:00,140
15,2026-03-13 15:00:00,130
16,2026-03-13 16:00:00,140
17,2026-03-13 17:00:00,160


In [ ]:
baseline_df[baseline_df["baseline_run_flag"] == 1][["timestamp","carbon_g_per_kwh"]]

,timestamp,carbon_g_per_kwh
0,2026-03-13 00:00:00,340
1,2026-03-13 01:00:00,330
2,2026-03-13 02:00:00,320
3,2026-03-13 03:00:00,310
4,2026-03-13 04:00:00,300
5,2026-03-13 05:00:00,290
6,2026-03-13 06:00:00,280
7,2026-03-13 07:00:00,270


In [ ]:
optimized_df[optimized_df["run_flag"] == 1][["timestamp","carbon_g_per_kwh"]]

,timestamp,carbon_g_per_kwh
10,2026-03-13 10:00:00,200
11,2026-03-13 11:00:00,180
12,2026-03-13 12:00:00,160
13,2026-03-13 13:00:00,150
14,2026-03-13 14:00:00,140
15,2026-03-13 15:00:00,130
16,2026-03-13 16:00:00,140
17,2026-03-13 17:00:00,160


In [ ]:
comparison = compare_schedules(
    baseline_df=baseline_df,
    optimized_df=optimized_df,
    machine_watts=300
)

import pandas as pd
pd.DataFrame([comparison])

,baseline_cost,optimized_cost,cost_savings,cost_reduction_pct,baseline_carbon_kg,optimized_carbon_kg,carbon_savings_kg,carbon_reduction_pct
0,0.21,0.678,-0.468,-222.857143,0.732,0.378,0.354,48.360656


In [ ]:
optimized_deadline_df = optimize_schedule(
    forecast_df=forecast_df,
    compute_hours_required=8,
    objective="carbon",
    deadline="2026-03-13 17:00"
)

optimized_deadline_df[optimized_deadline_df["run_flag"] == 1][
    ["timestamp", "carbon_g_per_kwh", "eligible_flag", "run_flag"]
]

,timestamp,carbon_g_per_kwh,eligible_flag,run_flag
10,2026-03-13 10:00:00,200,1,1
11,2026-03-13 11:00:00,180,1,1
12,2026-03-13 12:00:00,160,1,1
13,2026-03-13 13:00:00,150,1,1
14,2026-03-13 14:00:00,140,1,1
15,2026-03-13 15:00:00,130,1,1
16,2026-03-13 16:00:00,140,1,1
17,2026-03-13 17:00:00,160,1,1


In [ ]:
optimized_tight_df = optimize_schedule(
    forecast_df=forecast_df,
    compute_hours_required=8,
    objective="carbon",
    deadline="2026-03-13 12:00"
)

optimized_tight_df[optimized_tight_df["run_flag"] == 1][
    ["timestamp", "carbon_g_per_kwh", "eligible_flag", "run_flag"]
]

,timestamp,carbon_g_per_kwh,eligible_flag,run_flag
5,2026-03-13 05:00:00,290,1,1
6,2026-03-13 06:00:00,280,1,1
7,2026-03-13 07:00:00,270,1,1
8,2026-03-13 08:00:00,220,1,1
9,2026-03-13 09:00:00,210,1,1
10,2026-03-13 10:00:00,200,1,1
11,2026-03-13 11:00:00,180,1,1
12,2026-03-13 12:00:00,160,1,1


In [ ]:
comparison_deadline = compare_schedules(
    baseline_df=baseline_df,
    optimized_df=optimized_deadline_df,
    machine_watts=300
)

import pandas as pd
pd.DataFrame([comparison_deadline])

,baseline_cost,optimized_cost,cost_savings,cost_reduction_pct,baseline_carbon_kg,optimized_carbon_kg,carbon_savings_kg,carbon_reduction_pct
0,0.21,0.678,-0.468,-222.857143,0.732,0.378,0.354,48.360656


In [ ]:
from src.scheduler import format_schedule

formatted_schedule_df = format_schedule(optimized_deadline_df)

formatted_schedule_df

,timestamp,eligible_flag,run_flag,recommended_action,price_per_kwh,carbon_g_per_kwh
0,2026-03-13 00:00:00,1,0,Wait,0.09,340
1,2026-03-13 01:00:00,1,0,Wait,0.08,330
2,2026-03-13 02:00:00,1,0,Wait,0.08,320
3,2026-03-13 03:00:00,1,0,Wait,0.07,310
4,2026-03-13 04:00:00,1,0,Wait,0.07,300
5,2026-03-13 05:00:00,1,0,Wait,0.08,290
6,2026-03-13 06:00:00,1,0,Wait,0.10,280
7,2026-03-13 07:00:00,1,0,Wait,0.13,270
8,2026-03-13 08:00:00,1,0,Wait,0.16,220
9,2026-03-13 09:00:00,1,0,Wait,0.18,210


In [ ]:
formatted_schedule_df[formatted_schedule_df["recommended_action"] == "Run"]

,timestamp,eligible_flag,run_flag,recommended_action,price_per_kwh,carbon_g_per_kwh
10,2026-03-13 10:00:00,1,1,Run,0.20,200
11,2026-03-13 11:00:00,1,1,Run,0.22,180
12,2026-03-13 12:00:00,1,1,Run,0.25,160
13,2026-03-13 13:00:00,1,1,Run,0.27,150
14,2026-03-13 14:00:00,1,1,Run,0.30,140
15,2026-03-13 15:00:00,1,1,Run,0.32,130
16,2026-03-13 16:00:00,1,1,Run,0.34,140
17,2026-03-13 17:00:00,1,1,Run,0.36,160


In [ ]:
from src.inputs import WorkloadInput

sample_input = WorkloadInput(
    zip_code="93106",
    compute_hours_required=8,
    deadline="2026-03-13 17:00",
    objective="carbon",
    machine_watts=300,
)

sample_input

WorkloadInput(zip_code='93106', compute_hours_required=8, deadline='2026-03-13 17:00', objective='carbon', machine_watts=300)

In [ ]:
bad_input = WorkloadInput(
    zip_code="9310",
    compute_hours_required=8,
    deadline="2026-03-13 17:00",
    objective="carbon",
    machine_watts=300,
)

ValueError: zip_code must be a 5-digit string

In [ ]:
from src.mapper import load_zip_region_map, map_zip_to_region

mapping_path = PROJECT_ROOT / "data" / "raw" / "zip_to_region_sample.csv"
mapping_df = load_zip_region_map(mapping_path)

mapping_df

,zip_code,region
0,93106,CAISO
1,90001,CAISO
2,94105,CAISO
3,73301,ERCOT
4,10001,NYISO
5,60601,PJM


In [ ]:
region = map_zip_to_region("93106", mapping_df)
region

'CAISO'

In [ ]:
map_zip_to_region("99999", mapping_df)

ValueError: No region mapping found for ZIP code: 99999

In [ ]:
from pathlib import Path
from src.inputs import WorkloadInput
from src.pipeline import run_util_pipeline

PROJECT_ROOT = Path.cwd().resolve().parent

workload = WorkloadInput(
    zip_code="93106",
    compute_hours_required=8,
    deadline="2026-03-13 17:00",
    objective="carbon",
    machine_watts=300,
)

result = run_util_pipeline(
    workload_input=workload,
    mapping_path=PROJECT_ROOT / "data" / "raw" / "zip_to_region_sample.csv",
    carbon_path=PROJECT_ROOT / "data" / "raw" / "sample_carbon_forecast.csv",
    price_path=PROJECT_ROOT / "data" / "raw" / "sample_price_forecast.csv",
)

print("Region:", result["region"])
print("Metrics:", result["metrics"])
result["schedule"].head(24)

Region: CAISO
Metrics: {'baseline_cost': np.float64(0.21000000000000002), 'optimized_cost': np.float64(0.6779999999999999), 'cost_savings': np.float64(-0.4679999999999999), 'cost_reduction_pct': np.float64(-222.85714285714278), 'baseline_carbon_kg': np.float64(0.732), 'optimized_carbon_kg': np.float64(0.378), 'carbon_savings_kg': np.float64(0.354), 'carbon_reduction_pct': np.float64(48.36065573770492)}


,timestamp,eligible_flag,run_flag,recommended_action,price_per_kwh,carbon_g_per_kwh
0,2026-03-13 00:00:00,1,0,Wait,0.09,340
1,2026-03-13 01:00:00,1,0,Wait,0.08,330
2,2026-03-13 02:00:00,1,0,Wait,0.08,320
3,2026-03-13 03:00:00,1,0,Wait,0.07,310
4,2026-03-13 04:00:00,1,0,Wait,0.07,300
5,2026-03-13 05:00:00,1,0,Wait,0.08,290
6,2026-03-13 06:00:00,1,0,Wait,0.10,280
7,2026-03-13 07:00:00,1,0,Wait,0.13,270
8,2026-03-13 08:00:00,1,0,Wait,0.16,220
9,2026-03-13 09:00:00,1,0,Wait,0.18,210


In [ ]:
print((PROJECT_ROOT / "data" / "raw" / "zip_to_region_sample.csv").exists())
print((PROJECT_ROOT / "data" / "raw" / "sample_carbon_forecast.csv").exists())
print((PROJECT_ROOT / "data" / "raw" / "sample_price_forecast.csv").exists())

True
True
True


In [ ]:
# --- Notebook setup (ensures project root + paths work) ---
import sys
from pathlib import Path

NOTEBOOK_DIR = Path.cwd().resolve()

if NOTEBOOK_DIR.name == "notebooks":
    PROJECT_ROOT = NOTEBOOK_DIR.parent
else:
    PROJECT_ROOT = NOTEBOOK_DIR

if str(PROJECT_ROOT) not in sys.path:
    sys.path.append(str(PROJECT_ROOT))

print("Project root:", PROJECT_ROOT)

# --- Imports ---
from src.inputs import WorkloadInput
from src.pipeline import run_util_pipeline

# --- Build input ---
workload = WorkloadInput(
    zip_code="93106",
    compute_hours_required=8,
    deadline="2026-03-13 17:00",
    objective="carbon",
    machine_watts=300,
)

# --- Run pipeline ---
result = run_util_pipeline(
    workload_input=workload,
    mapping_path=PROJECT_ROOT / "data" / "raw" / "zip_to_region_sample.csv",
    carbon_path=PROJECT_ROOT / "data" / "raw" / "sample_carbon_forecast.csv",
    price_path=PROJECT_ROOT / "data" / "raw" / "sample_price_forecast.csv",
)

# --- Output results ---
print("Region:", result["region"])
print("Metrics:", result["metrics"])

result["schedule"].head(24)

Project root: C:\dev\util
Region: CAISO
Metrics: {'baseline_cost': np.float64(0.21000000000000002), 'optimized_cost': np.float64(0.6779999999999999), 'cost_savings': np.float64(-0.4679999999999999), 'cost_reduction_pct': np.float64(-222.85714285714278), 'baseline_carbon_kg': np.float64(0.732), 'optimized_carbon_kg': np.float64(0.378), 'carbon_savings_kg': np.float64(0.354), 'carbon_reduction_pct': np.float64(48.36065573770492)}


,timestamp,eligible_flag,run_flag,recommended_action,price_per_kwh,carbon_g_per_kwh
0,2026-03-13 00:00:00,1,0,Wait,0.09,340
1,2026-03-13 01:00:00,1,0,Wait,0.08,330
2,2026-03-13 02:00:00,1,0,Wait,0.08,320
3,2026-03-13 03:00:00,1,0,Wait,0.07,310
4,2026-03-13 04:00:00,1,0,Wait,0.07,300
5,2026-03-13 05:00:00,1,0,Wait,0.08,290
6,2026-03-13 06:00:00,1,0,Wait,0.10,280
7,2026-03-13 07:00:00,1,0,Wait,0.13,270
8,2026-03-13 08:00:00,1,0,Wait,0.16,220
9,2026-03-13 09:00:00,1,0,Wait,0.18,210


In [ ]:
print("===== UTIL DEMO =====")

print("Region:", result["region"])

print("\nMETRICS")
for k, v in result["metrics"].items():
    print(f"{k}: {v}")

print("\nSCHEDULE PREVIEW")
display(result["schedule"].head(24))

===== UTIL DEMO =====
Region: CAISO

METRICS
baseline_cost: 0.21000000000000002
optimized_cost: 0.6779999999999999
cost_savings: -0.4679999999999999
cost_reduction_pct: -222.85714285714278
baseline_carbon_kg: 0.732
optimized_carbon_kg: 0.378
carbon_savings_kg: 0.354
carbon_reduction_pct: 48.36065573770492

SCHEDULE PREVIEW


,timestamp,eligible_flag,run_flag,recommended_action,price_per_kwh,carbon_g_per_kwh
0,2026-03-13 00:00:00,1,0,Wait,0.09,340
1,2026-03-13 01:00:00,1,0,Wait,0.08,330
2,2026-03-13 02:00:00,1,0,Wait,0.08,320
3,2026-03-13 03:00:00,1,0,Wait,0.07,310
4,2026-03-13 04:00:00,1,0,Wait,0.07,300
5,2026-03-13 05:00:00,1,0,Wait,0.08,290
6,2026-03-13 06:00:00,1,0,Wait,0.10,280
7,2026-03-13 07:00:00,1,0,Wait,0.13,270
8,2026-03-13 08:00:00,1,0,Wait,0.16,220
9,2026-03-13 09:00:00,1,0,Wait,0.18,210


In [ ]:
from pathlib import Path
from src.mapper import load_zip_region_map, map_zip_to_region

mapping_df = load_zip_region_map(
    PROJECT_ROOT / "data" / "raw" / "zip_to_region_sample.csv"
)

print(mapping_df.head())

print("93106 region:", map_zip_to_region("93106", mapping_df))

  zip_code region
0    93106  CAISO
1    90001  CAISO
2    94105  CAISO
3    73301  ERCOT
4    10001  NYISO
93106 region: CAISO


In [ ]:
map_zip_to_region("93106", mapping_df)

'CAISO'

In [ ]:
from services.watttime_service import get_watttime_forecast

df = get_watttime_forecast(region="CAISO_NORTH")
print(df.head())
print(df.columns)


FORECAST STATUS: 200
FORECAST CONTENT TYPE: application/json
FORECAST URL: https://api.watttime.org/v3/forecast?region=CAISO_NORTH&signal_type=co2_moer
FORECAST RESPONSE PREVIEW: {"data":[{"point_time":"2026-03-14T03:10:00+00:00","value":1000.6},{"point_time":"2026-03-14T03:15:00+00:00","value":1000.6},{"point_time":"2026-03-14T03:20:00+00:00","value":997.1},{"point_time":"2026-03-14T03:25:00+00:00","value":990.1},{"point_time":"2026-03-14T03:30:00+00:00","value":992.2},{"point_time":"2026-03-14T03:35:00+00:00","value":965.0},{"point_time":"2026-03-14T03:40:00+00:00","value":982.2},{"point_time":"2026-03-14T03:45:00+00:00","value":983.5},{"point_time":"2026-03-14T03:50:0
                  timestamp  carbon_g_per_kwh
0 2026-03-14 03:10:00+00:00            1000.6
1 2026-03-14 03:15:00+00:00            1000.6
2 2026-03-14 03:20:00+00:00             997.1
3 2026-03-14 03:25:00+00:00             990.1
4 2026-03-14 03:30:00+00:00             992.2
Index(['timestamp', 'carbon_g_per_kwh'], dty

In [ ]:
from src.pipeline import run_util_pipeline
from src.inputs import WorkloadInput

workload = WorkloadInput(
    zip_code="93106",
    compute_hours_required=8,
    deadline="2026-03-14 08:00",
    objective="carbon",
    machine_watts=300,
)

result = run_util_pipeline(
    workload_input=workload,
    mapping_path="data/raw/zip_to_region_sample.csv",
    carbon_path="data/raw/sample_carbon_forecast.csv",
    price_path="data/raw/sample_price_forecast.csv",
    forecast_mode="demo",
)

print(result["forecast"].head())
print(result["metrics"])


TypeError: run_util_pipeline() got an unexpected keyword argument 'forecast_mode'

In [4]:
from src.pipeline import run_util_pipeline
import inspect

print(inspect.signature(run_util_pipeline))

(workload_input: 'Any', mapping_path: 'str | Path', carbon_path: 'str | Path | None' = None, price_path: 'str | Path | None' = None, forecast_mode: 'str' = 'demo') -> 'dict[str, Any]'


In [6]:
from src.pipeline import run_util_pipeline
print(run_util_pipeline.__code__.co_filename)

C:\dev\util\src\pipeline.py


In [15]:
import os
print(os.getcwd())

c:\dev\util\notebooks


In [17]:
from pathlib import Path
from src.pipeline import run_util_pipeline
from src.inputs import WorkloadInput

project_root = Path.cwd().parent
print("Project root:", project_root)

workload = WorkloadInput(
    zip_code="93106",
    compute_hours_required=8,
    deadline="2026-03-14 08:00",
    objective="carbon",
    machine_watts=300,
)

result = run_util_pipeline(
    workload_input=workload,
    mapping_path=project_root / "data" / "raw" / "zip_to_region_sample.csv",
    carbon_path=project_root / "data" / "raw" / "sample_carbon_forecast.csv",
    price_path=project_root / "data" / "raw" / "sample_price_forecast.csv",
    forecast_mode="demo",
)

print(result["forecast"].head())
print(result["metrics"])

Project root: c:\dev\util
            timestamp  carbon_g_per_kwh  price_per_kwh
0 2026-03-13 00:00:00               340           0.09
1 2026-03-13 01:00:00               330           0.08
2 2026-03-13 02:00:00               320           0.08
3 2026-03-13 03:00:00               310           0.07
4 2026-03-13 04:00:00               300           0.07
{'baseline_cost': np.float64(0.21000000000000002), 'optimized_cost': np.float64(0.6779999999999999), 'cost_savings': np.float64(-0.4679999999999999), 'cost_reduction_pct': np.float64(-222.85714285714278), 'baseline_carbon_kg': np.float64(0.732), 'optimized_carbon_kg': np.float64(0.378), 'carbon_savings_kg': np.float64(0.354), 'carbon_reduction_pct': np.float64(48.36065573770492)}


In [19]:
from pathlib import Path
from src.pipeline import run_util_pipeline
from src.inputs import WorkloadInput

project_root = Path.cwd().parent

workload = WorkloadInput(
    zip_code="93106",
    compute_hours_required=8,
    deadline="2026-03-14 08:00",
    objective="carbon",
    machine_watts=300,
)

result = run_util_pipeline(
    workload_input=workload,
    mapping_path=project_root / "data" / "raw" / "zip_to_region_sample.csv",
    forecast_mode="live_carbon",
)

print(result["forecast"].head())
print(result["metrics"])

FORECAST STATUS: 400
FORECAST CONTENT TYPE: application/json
FORECAST URL: https://api.watttime.org/v3/forecast?region=CAISO&signal_type=co2_moer
FORECAST RESPONSE PREVIEW: {"error":"INVALID_REGION","message":"You must provide a valid region. You may query /region-from-loc to determine the region for a given location","docs":"https://docs.watttime.org/#tag/GET-Regions-and-Maps/operation/get_reg_loc_v3_region_from_loc_get"}


HTTPError: 400 Client Error: Bad Request for url: https://api.watttime.org/v3/forecast?region=CAISO&signal_type=co2_moer

In [20]:
from src.data_fetcher import build_live_carbon_forecast_table
import inspect

print(build_live_carbon_forecast_table.__code__.co_filename)
print(inspect.getsource(build_live_carbon_forecast_table))

C:\dev\util\src\data_fetcher.py
def build_live_carbon_forecast_table(
    region: str,
    placeholder_price_per_kwh: float = 0.15,
) -> pd.DataFrame:
    """
    Build a forecast table using live carbon forecast data from WattTime
    and a placeholder electricity price.

    IMPORTANT PROTOTYPE NOTE:
    The current ZIP-to-region mapper returns internal prototype regions
    like "CAISO", but the WattTime forecast endpoint currently needs a
    WattTime-compatible region such as "CAISO_NORTH".

    For the prototype MVP, we intentionally hardcode the WattTime region
    to CAISO_NORTH so live carbon mode remains stable while we defer
    ZIP -> lat/lon -> WattTime region lookup to a later phase.

    Returns columns:
    - timestamp
    - carbon_g_per_kwh
    - price_per_kwh
    """
    watttime_region = "CAISO_NORTH"

    carbon_df = get_watttime_forecast(watttime_region)

    required_columns = {"timestamp", "carbon_g_per_kwh"}
    if not required_columns.issubset(carbon_df.columns

In [29]:
from pathlib import Path

path = Path(r"C:\dev\util\src\data_fetcher.py")

with open(path, "r", encoding="utf-8") as f:
    lines = f.readlines()

for i in range(100, 116):
    print(f"{i+1}: {lines[i].rstrip()}")

101:     """
102:     _ = region  # kept for future compatibility
103:     watttime_region = "CAISO_NORTH"
104: 
105:     carbon_df = get_watttime_forecast(watttime_region)
106: 
107:     required_columns = {"timestamp", "carbon_g_per_kwh"}
108:     if not required_columns.issubset(carbon_df.columns):
109:         raise ValueError(
110:             f"WattTime forecast must contain columns: {required_columns}"
111:         )
112: 
113:     carbon_df = carbon_df.copy()
114:     carbon_df["timestamp"] = pd.to_datetime(carbon_df["timestamp"])
115:     carbon_df["price_per_kwh"] = placeholder_price_per_kwh
116: 


In [30]:
from src.data_fetcher import build_live_carbon_forecast_table
import inspect

print(inspect.getsource(build_live_carbon_forecast_table))

def build_live_carbon_forecast_table(
    region: str,
    placeholder_price_per_kwh: float = 0.15,
) -> pd.DataFrame:
    """
    Build a forecast table using live carbon forecast data from WattTime
    and a placeholder electricity price.

    IMPORTANT PROTOTYPE NOTE:
    The current ZIP-to-region mapper returns internal prototype regions
    like "CAISO", but the WattTime forecast endpoint currently needs a
    WattTime-compatible region such as "CAISO_NORTH".

    For the prototype MVP, we intentionally hardcode the WattTime region
    to CAISO_NORTH so live carbon mode remains stable while we defer
    ZIP -> lat/lon -> WattTime region lookup to a later phase.

    Returns columns:
    - timestamp
    - carbon_g_per_kwh
    - price_per_kwh
    """
    _ = region
    watttime_region = "CAISO_NORTH"

    carbon_df = get_watttime_forecast(watttime_region)

    required_columns = {"timestamp", "carbon_g_per_kwh"}
    if not required_columns.issubset(carbon_df.columns):
        raise 

In [4]:
from src.data_fetcher import build_live_carbon_forecast_table

df = build_live_carbon_forecast_table(region="CAISO")
print(df.head())
print(df.columns)

FORECAST STATUS: 200
FORECAST CONTENT TYPE: application/json
FORECAST URL: https://api.watttime.org/v3/forecast?region=CAISO_NORTH&signal_type=co2_moer
FORECAST RESPONSE PREVIEW: {"data":[{"point_time":"2026-03-14T03:45:00+00:00","value":997.2},{"point_time":"2026-03-14T03:50:00+00:00","value":997.2},{"point_time":"2026-03-14T03:55:00+00:00","value":997.0},{"point_time":"2026-03-14T04:00:00+00:00","value":993.9},{"point_time":"2026-03-14T04:05:00+00:00","value":992.1},{"point_time":"2026-03-14T04:10:00+00:00","value":992.4},{"point_time":"2026-03-14T04:15:00+00:00","value":992.9},{"point_time":"2026-03-14T04:20:00+00:00","value":988.9},{"point_time":"2026-03-14T04:25:00+
                  timestamp  carbon_g_per_kwh  price_per_kwh
0 2026-03-14 03:45:00+00:00             997.2           0.15
1 2026-03-14 03:50:00+00:00             997.2           0.15
2 2026-03-14 03:55:00+00:00             997.0           0.15
3 2026-03-14 04:00:00+00:00             993.9           0.15
4 2026-03-14 04

In [1]:
import sys
from pathlib import Path

project_root = Path.cwd().parent
if str(project_root) not in sys.path:
    sys.path.append(str(project_root))

from src.pipeline import run_util_pipeline
from src.inputs import WorkloadInput

workload = WorkloadInput(
    zip_code="93106",
    compute_hours_required=8,
    deadline="2026-03-14 08:00",
    objective="carbon",
    machine_watts=300,
)

result = run_util_pipeline(
    workload_input=workload,
    mapping_path=project_root / "data" / "raw" / "zip_to_region_sample.csv",
    forecast_mode="live_carbon",
)

print(result["forecast"].head())
print(result["metrics"])

FORECAST STATUS: 200
FORECAST CONTENT TYPE: application/json
FORECAST URL: https://api.watttime.org/v3/forecast?region=CAISO_NORTH&signal_type=co2_moer
FORECAST RESPONSE PREVIEW: {"data":[{"point_time":"2026-03-14T03:45:00+00:00","value":997.2},{"point_time":"2026-03-14T03:50:00+00:00","value":997.2},{"point_time":"2026-03-14T03:55:00+00:00","value":997.0},{"point_time":"2026-03-14T04:00:00+00:00","value":993.9},{"point_time":"2026-03-14T04:05:00+00:00","value":992.1},{"point_time":"2026-03-14T04:10:00+00:00","value":992.4},{"point_time":"2026-03-14T04:15:00+00:00","value":992.9},{"point_time":"2026-03-14T04:20:00+00:00","value":988.9},{"point_time":"2026-03-14T04:25:00+
            timestamp  carbon_g_per_kwh  price_per_kwh
0 2026-03-14 03:45:00             997.2           0.15
1 2026-03-14 03:50:00             997.2           0.15
2 2026-03-14 03:55:00             997.0           0.15
3 2026-03-14 04:00:00             993.9           0.15
4 2026-03-14 04:05:00             992.1      

In [2]:
from src.data_fetcher import build_live_carbon_forecast_table

df = build_live_carbon_forecast_table(region="CAISO")
print(df.head())
print(df["timestamp"].dtype)

FORECAST STATUS: 200
FORECAST CONTENT TYPE: application/json
FORECAST URL: https://api.watttime.org/v3/forecast?region=CAISO_NORTH&signal_type=co2_moer
FORECAST RESPONSE PREVIEW: {"data":[{"point_time":"2026-03-14T03:45:00+00:00","value":997.2},{"point_time":"2026-03-14T03:50:00+00:00","value":997.2},{"point_time":"2026-03-14T03:55:00+00:00","value":997.0},{"point_time":"2026-03-14T04:00:00+00:00","value":993.9},{"point_time":"2026-03-14T04:05:00+00:00","value":992.1},{"point_time":"2026-03-14T04:10:00+00:00","value":992.4},{"point_time":"2026-03-14T04:15:00+00:00","value":992.9},{"point_time":"2026-03-14T04:20:00+00:00","value":988.9},{"point_time":"2026-03-14T04:25:00+
            timestamp  carbon_g_per_kwh  price_per_kwh
0 2026-03-14 03:45:00             997.2           0.15
1 2026-03-14 03:50:00             997.2           0.15
2 2026-03-14 03:55:00             997.0           0.15
3 2026-03-14 04:00:00             993.9           0.15
4 2026-03-14 04:05:00             992.1      

In [3]:
from src.pipeline import run_util_pipeline
from src.inputs import WorkloadInput

workload = WorkloadInput(
    zip_code="93106",
    compute_hours_required=8,
    deadline="2026-03-14 08:00",
    objective="carbon",
    machine_watts=300,
)

result = run_util_pipeline(
    workload_input=workload,
    mapping_path=project_root / "data" / "raw" / "zip_to_region_sample.csv",
    forecast_mode="live_carbon",
)

print(result["forecast"].head())
print(result["metrics"])

FORECAST STATUS: 200
FORECAST CONTENT TYPE: application/json
FORECAST URL: https://api.watttime.org/v3/forecast?region=CAISO_NORTH&signal_type=co2_moer
FORECAST RESPONSE PREVIEW: {"data":[{"point_time":"2026-03-14T03:45:00+00:00","value":997.2},{"point_time":"2026-03-14T03:50:00+00:00","value":997.2},{"point_time":"2026-03-14T03:55:00+00:00","value":997.0},{"point_time":"2026-03-14T04:00:00+00:00","value":993.9},{"point_time":"2026-03-14T04:05:00+00:00","value":992.1},{"point_time":"2026-03-14T04:10:00+00:00","value":992.4},{"point_time":"2026-03-14T04:15:00+00:00","value":992.9},{"point_time":"2026-03-14T04:20:00+00:00","value":988.9},{"point_time":"2026-03-14T04:25:00+
            timestamp  carbon_g_per_kwh  price_per_kwh
0 2026-03-14 03:45:00             997.2           0.15
1 2026-03-14 03:50:00             997.2           0.15
2 2026-03-14 03:55:00             997.0           0.15
3 2026-03-14 04:00:00             993.9           0.15
4 2026-03-14 04:05:00             992.1      